# Demo 2: Grover on 3 qubits
Slide 14 (Demo 2 of 3): after slide 13 (*How fast can Grover crack a password?*), before slide 16 (*Why is the speedup quadratic?*).
8 possible "passwords", one is correct.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from demo_utils import plot_probs, plot_counts, load_ibm, GREEN, LIGHT

n = 3
dev = qml.device("default.qubit")

### Pick the secret
*Ask the room for a number from 0 to 7.*

In [ ]:
secret = 5
secret_bits = [int(b) for b in format(secret, f"0{n}b")]
print("secret:", secret, "in binary:", "".join(map(str, secret_bits)))

### The algorithm: superposition, then repeat oracle + diffusion

In [ ]:
@qml.qnode(dev)
def grover(iterations):
    for w in range(n):
        qml.Hadamard(wires=w)                        # try all 8 at once
    for _ in range(iterations):
        qml.FlipSign(secret_bits, wires=range(n))    # oracle: flip the sign of the secret
        qml.GroverOperator(wires=range(n))           # diffusion: reflect about the mean
    return qml.probs(wires=range(n))

target = format(secret, f"0{n}b")
plot_probs(grover(0), title="0 iterations: every answer equally likely", highlight=target)
plot_probs(grover(2), title="2 iterations: the secret stands out", highlight=target)

### More is not better
The optimum is about π/4 · √8 ≈ 2 iterations. After that the state rotates *past* the answer.

In [ ]:
ks = np.arange(0, 9)
p_sim = [grover(int(k))[secret] for k in ks]

theta = np.arcsin(1 / np.sqrt(2 ** n))
kk = np.linspace(0, 8, 200)
p_theory = np.sin((2 * kk + 1) * theta) ** 2

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(kk, p_theory, color=LIGHT, lw=2, label="theory: sin²((2k+1)θ)")
ax.scatter(ks, p_sim, color=GREEN, s=50, zorder=3, label="simulation")
ax.set_xlabel("Grover iterations k")
ax.set_ylabel("P(secret)")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, loc="upper center")
ax.set_title("Success probability rotates, it doesn't just grow")
plt.show()

### Sampling like a real QPU: 1,000 shots

In [ ]:
@qml.set_shots(1000)
@qml.qnode(dev)
def grover_shots(iterations):
    for w in range(n):
        qml.Hadamard(wires=w)
    for _ in range(iterations):
        qml.FlipSign(secret_bits, wires=range(n))
        qml.GroverOperator(wires=range(n))
    return qml.counts(wires=range(n))

plot_counts(grover_shots(2), title="2 iterations, 1,000 shots", highlight=target)

### Real IBM QPU (secret = 5, sent at the start of the talk)

In [ ]:
counts, backend, date = load_ibm("grover")
plot_counts(counts, title=f"Grover, secret 101, on {backend} ({date})", highlight="101")